# Constroi o Dataset das Eleições Somente com os Dados de SJDR

In [34]:
import pandas as pd
import numpy as np
import unicodedata
import pathlib

In [35]:
data_path = pathlib.Path("./data")

# Arquivos de boletim de urna de cada um dos turnos
csv_1t = data_path / "bweb_1t_MG_051020261403.csv"
csv_2t = data_path / "bweb_2t_MG_261020260742.csv"

# Arquivo com a localização de cada um dos locais de votação
csv_locais = data_path / "eleitorado_local_votacao_2026_MG.csv"

In [36]:
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return texto
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto.upper().replace('-', ' ').strip()

In [37]:
municipio_alvo = 'SAO JOAO DEL REI'

chunk_size = 100000

colunas_int = [
    'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NR_TURNO', 'CD_ELEICAO', 
    'CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'CD_CARGO', 
    'NR_VOTAVEL', 'QT_VOTOS', 'NR_LOCAL_VOTACAO', 'SQ_CANDIDATO'
]

colunas_datas = ['DT_GERACAO', 'DT_ELEICAO']

valores_nulos = ['#NULO', '#NE', '#NULO#', '#NE#', -1, -3, '-1', '-3']

In [38]:
# Retorna df com as informacoes do municipio 
def separa_municipios(arquivo_origem, municipio:str):
    dfs_municipio = []

    for chunk in pd.read_csv(arquivo_origem, encoding='latin1', sep=';', chunksize=chunk_size, low_memory=False):
        chunk = chunk.replace(valores_nulos, np.nan)
        chunk['NM_MUNICIPIO_CLEAN'] = chunk['NM_MUNICIPIO'].astype(str).apply(normalizar_texto)
        
        # Conversão de Tipos de Dados
        for col in colunas_datas:
            if col in chunk.columns:
                chunk[col] = pd.to_datetime(chunk[col], format='%d/%m/%Y', errors='coerce')
                
        for col in colunas_int:
            if col in chunk.columns:
                chunk[col] = pd.to_numeric(chunk[col], errors='coerce').astype('Int64')
                
        df_mun = chunk[chunk['NM_MUNICIPIO_CLEAN'] == municipio].copy()
        
        if not df_mun.empty:
            df_mun = df_mun.drop(columns=['NM_MUNICIPIO_CLEAN'])
            dfs_municipio.append(df_mun) 
    
    if dfs_municipio:
        return pd.concat(dfs_municipio, ignore_index=True)
    
    return pd.DataFrame() 

**Seleciona apenas as zonas eleitorais correspondentes a SJDR**

In [39]:
df_municipio = pd.concat(
    [separa_municipios(csv_1t, municipio_alvo), 
    separa_municipios(csv_2t, municipio_alvo)],
    ignore_index=True
)

df_municipio.head()

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_PLEITO,DT_PLEITO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,...,CD_FLASHCARD_URNA_EFETIVADA,DT_CARGA_URNA_EFETIVADA,DS_CARGO_PERGUNTA_SECAO,DS_SECOES_AGREGADAS,DT_ABERTURA,DT_ENCERRAMENTO,QT_ELEI_BIOM_SEM_HABILITACAO,DT_EMISSAO_BU,NR_JUNTA_APURADORA,NR_TURMA_APURADORA
0,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6259,Eleições Gerais Estaduais 2026,...,5E69193D,2026-09-24 11:36:00,7 - 308,NaN,2026-10-04 08:16:53,2026-10-04 17:32:59,36,2026-10-04 17:36:06,NaN,NaN
1,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6259,Eleições Gerais Estaduais 2026,...,5E69193D,2026-09-23 11:42:00,5 - 221,NaN,2026-10-04 08:00:01,2026-10-04 17:01:03,43,2026-10-04 17:02:39,NaN,NaN
2,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6259,Eleições Gerais Estaduais 2026,...,44CD4F53,2026-09-23 16:41:00,6 - 152,NaN,2026-10-04 08:00:01,2026-10-04 17:05:04,14,2026-10-04 17:06:44,NaN,NaN
3,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6259,Eleições Gerais Estaduais 2026,...,5E69193D,2026-09-23 11:42:00,6 - 221,NaN,2026-10-04 08:00:01,2026-10-04 17:01:03,43,2026-10-04 17:02:39,NaN,NaN
4,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6259,Eleições Gerais Estaduais 2026,...,44CD4F53,2026-09-23 17:27:00,6 - 155,NaN,2026-10-04 08:00:01,2026-10-04 17:01:04,29,2026-10-04 17:03:03,NaN,NaN


**Seleciona somente os registros correspondentes as eleicoes presidenciais**

In [40]:
df_municipio = df_municipio[df_municipio['CD_CARGO_PERGUNTA'] == 1]
print(df_municipio['CD_CARGO_PERGUNTA'].nunique())

1


**Carrega dataset com locais e cruza com os dados dos boletins de urna**

In [41]:
df_locais = pd.read_csv(csv_locais, encoding='latin1', sep=';', low_memory=False)

In [42]:
merge_keys = [
    "NR_TURNO",
    "SG_UF",
    "CD_MUNICIPIO",
    "NR_ZONA",
    "NR_SECAO",
    "NR_LOCAL_VOTACAO"
]

# Coloca só as colunas extras pra não gerar colunas duplicadas
df_locais_not_in_df_municipio = [col for col in df_locais.columns if col not in df_municipio.columns]

df_municipio = pd.merge(
    df_municipio,
    df_locais[merge_keys + df_locais_not_in_df_municipio],
    how='left',
    on=merge_keys,
)

# Corrige problema dos locais sem lat lon por conta de mudança do código do local de votação  
chaves_secao = ["CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"]
locais_por_secao = (
    df_locais[chaves_secao + df_locais_not_in_df_municipio]
    .dropna(subset=["NR_LATITUDE", "NR_LONGITUDE"])
    .drop_duplicates(subset=chaves_secao)
    .set_index(chaves_secao)
)

df_municipio = df_municipio.set_index(chaves_secao)
df_municipio.update(locais_por_secao, overwrite=False)
df_municipio = df_municipio.reset_index()

**Salva o arquivo**

In [43]:
df_municipio.to_csv(data_path / 'votacao_sjdr.csv', index=False, sep=',', encoding='utf-8-sig')